# UCSD Greynet Telescope — Data Access & FlowTuple Format Check

First look at the **SDSC greynet** data, for the greynet ↔ darknet comparison project.

A **greynet** interleaves instrumented-but-unused addresses *within* a live production
network (here: SDSC's), whereas a classic darknet like the UCSD Network Telescope
monitors one large, fully unused block. Unsolicited traffic reaching greynet addresses
arrives despite the surrounding network being real and active — which is what makes
comparing the two telescopes interesting.

This notebook runs **on Expanse** (a plain Jupyter session is enough — no Spark yet).
It is **read-only**. It:

1. loads S3 credentials from `/expanse/lustre/projects/sdp178/kmok/.ucsdnts3greynet.env`,
2. connects to the object store and lists the two greynet containers,
   `sdscgreynetlist` and `telescope-sdscnt-pcap-live`,
3. byte-sniffs sample objects (never trusting file extensions), and
4. for Avro objects, reads the embedded schema and first records to **confirm the data
   is in FlowTuple format** — the record format the darknet module
   (`nids-ucsdnt-expanse`) reads: `time`, `src_ip`, `packet_cnt`, `prefix2asn`,
   `netacq_country`, one file per 5-minute collection window.

Two assumptions this notebook makes and surfaces rather than hides:

- **Endpoint**: the same CAIDA S3 gateway the darknet FlowTuple data uses
  (`https://hermes.caida.org`) — unless the env file names one, which then wins.
- **Env variable names**: unknown in advance, so §2 parses the file and auto-detects
  the access / secret / endpoint variables, printing only the variable **names** it
  found (never the values).

## 0 · Dependencies

`boto3` talks to the object store, `fastavro` reads Avro object container files, and
`cramjam` covers the snappy codec in case the Avro blocks use it (FlowTuple files are
usually deflate). Each is imported first and only pip-installed (`--user`) if missing,
so this cell is a no-op on an image that already ships them.

In [ ]:
import importlib
import site
import subprocess
import sys


def ensure(pkg, module=None):
    """Import `module` (default: pkg), pip-installing --user first if missing."""
    module = module or pkg
    try:
        importlib.import_module(module)
        print(f"{pkg:<10} OK")
        return
    except ImportError:
        pass
    print(f"{pkg:<10} missing — installing with pip --user ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--user", "--quiet", pkg])
    if site.getusersitepackages() not in sys.path:
        sys.path.append(site.getusersitepackages())
    importlib.invalidate_caches()
    importlib.import_module(module)
    print(f"{pkg:<10} installed")


ensure("boto3")
ensure("fastavro")
ensure("cramjam")
ensure("pandas")

## 1 · Configuration

Everything tunable lives here. `telescope-sdscnt-pcap-live` is fed by a live telescope
and may hold a very large number of objects, so the full listing is capped
(`MAX_KEYS_PER_CONTAINER`); the delimiter listing in §3 still shows the container's
overall layout cheaply.

In [ ]:
ENV_FILE = "/expanse/lustre/projects/sdp178/kmok/.ucsdnts3greynet.env"

CONTAINERS = ["sdscgreynetlist", "telescope-sdscnt-pcap-live"]

# Same gateway the darknet FlowTuple module (nids-ucsdnt-expanse) uses.
# An endpoint-looking variable in ENV_FILE overrides it; ENDPOINT_OVERRIDE
# here overrides both.
DEFAULT_ENDPOINT = "https://hermes.caida.org"
ENDPOINT_OVERRIDE = None

MAX_KEYS_PER_CONTAINER = 2000  # listing cap for this first look
N_SAMPLE_OBJECTS = 3           # objects per container to byte-sniff (first/middle/last)
N_AVRO_RECORDS = 5             # records to decode from each Avro sample

## 2 · Load credentials

Parses the shell-style env file (`export KEY=value`, quotes, `#` comments) and
auto-detects which variable is the access key, the secret key, and (optionally) the
endpoint. Secret **values are never printed** — only the variable names, so a wrong
guess is easy to spot and fix. If the file is unreadable it falls back to variables
already exported in the kernel's environment (e.g. via `.bashrc`).

In [ ]:
import os
import pathlib
import re


def parse_env_file(path):
    """Parse a shell-style env file: optional `export`, KEY=value, quotes, # comments."""
    env = {}
    for line in pathlib.Path(path).read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        line = re.sub(r"^export\s+", "", line)
        m = re.match(r"([A-Za-z_][A-Za-z0-9_]*)\s*=\s*(.*)$", line)
        if not m:
            continue
        key, val = m.group(1), m.group(2).strip()
        if len(val) >= 2 and val[0] == val[-1] and val[0] in "\"'":
            val = val[1:-1]
        env[key] = val
    return env


def pick(env, *patterns):
    """First env key matching any regex, tried in order (case-insensitive)."""
    for pat in patterns:
        hits = [k for k in env if re.search(pat, k, re.I)]
        if hits:
            return hits[0]
    return None


try:
    env = parse_env_file(ENV_FILE)
    print(f"parsed {ENV_FILE}")
except OSError as exc:
    print(f"could not read {ENV_FILE} ({exc}); falling back to kernel environment")
    env = {k: v for k, v in os.environ.items() if re.search(r"s3|greynet", k, re.I)}

print("variables found:", ", ".join(sorted(env)) or "(none)")

access_var = pick(env, r"access.?key.?id", r"access.?key", r"access")
secret_var = pick(env, r"secret.?access.?key", r"secret.?key", r"secret")
endpoint_var = pick(env, r"endpoint", r"\burl\b", r"host")

assert access_var and secret_var, (
    "Could not identify access/secret variables among the names above — "
    "set access_var / secret_var manually in this cell."
)

ACCESS_KEY = env[access_var]
SECRET_KEY = env[secret_var]
S3_ENDPOINT_URL = (
    ENDPOINT_OVERRIDE
    or (env.get(endpoint_var) if endpoint_var else None)
    or DEFAULT_ENDPOINT
)

print(f"access key variable : {access_var}")
print(f"secret key variable : {secret_var}  (value not shown)")
print(
    f"endpoint            : {S3_ENDPOINT_URL}"
    + ("" if endpoint_var else "  (default — no endpoint named in env file)")
)

## 3 · Connect and list the containers

The client is configured exactly like the darknet module's: **signature v4** and
**path-style addressing**. For each container we take a delimiter listing first (the
top-level "directory" layout — the darknet bucket partitions as
`datasource=.../year=.../month=.../day=...`, and the greynet containers may do the
same), then a capped full listing for counts, sizes, and the last-modified range.

In [ ]:
import boto3
from botocore.config import Config

s3 = boto3.client(
    "s3",
    endpoint_url=S3_ENDPOINT_URL,
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    config=Config(signature_version="s3v4", s3={"addressing_style": "path"}),
)


def list_top_level(bucket):
    """Top-level 'directories' and loose objects, via a delimiter listing."""
    resp = s3.list_objects_v2(Bucket=bucket, Delimiter="/")
    prefixes = [p["Prefix"] for p in resp.get("CommonPrefixes", [])]
    loose = [o["Key"] for o in resp.get("Contents", [])]
    return prefixes, loose


def list_objects(bucket, max_keys):
    """Up to max_keys objects (dicts with Key/Size/LastModified) + truncation flag."""
    objs = []
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, PaginationConfig={"MaxItems": max_keys}):
        objs.extend(page.get("Contents", []))
    return objs, len(objs) >= max_keys


listings = {}
for bucket in CONTAINERS:
    print(f"=== {bucket} ===")
    try:
        prefixes, loose = list_top_level(bucket)
    except Exception as exc:
        print(f"  listing FAILED: {exc}")
        print("  (wrong endpoint? set ENDPOINT_OVERRIDE in section 1."
          " access denied? the key may be scoped differently — see section 2 output)\n")
        continue
    more = " ..." if len(prefixes) > 10 else ""
    print(f"  top-level prefixes ({len(prefixes)}): {prefixes[:10]}{more}")
    more = " ..." if len(loose) > 5 else ""
    print(f"  top-level objects  ({len(loose)}): {loose[:5]}{more}")

    objs, truncated = list_objects(bucket, MAX_KEYS_PER_CONTAINER)
    listings[bucket] = objs
    note = f" (capped at {MAX_KEYS_PER_CONTAINER} — more exist)" if truncated else " (complete)"
    print(f"  listed {len(objs)} objects{note}")
    if objs:
        total = sum(o["Size"] for o in objs)
        stamps = sorted(o["LastModified"] for o in objs)
        print(f"  total size of listed objects: {total / 2**20:,.1f} MiB")
        print(f"  last-modified range: {stamps[0]:%Y-%m-%d %H:%M} -> {stamps[-1]:%Y-%m-%d %H:%M} UTC")
        print(f"  first key: {objs[0]['Key']}")
        print(f"  last key : {objs[-1]['Key']}")
    print()

## 4 · Byte-sniff sample objects

File extensions lie (the previous module's streaming reader sniffed gzip magic bytes
for exactly that reason), so each sampled object is identified by its leading bytes:

| magic | format |
|---|---|
| `4F 62 6A 01` (`Obj\x01`) | **Avro** object container file |
| `D4 C3 B2 A1` / `A1 B2 C3 D4` | pcap (µs, LE/BE) |
| `4D 3C B2 A1` / `A1 B2 3C 4D` | pcap (ns, LE/BE) |
| `0A 0D 0D 0A` | pcapng |
| `1F 8B` | gzip — head is decompressed and re-sniffed |
| `28 B5 2F FD` | zstd |
| `04 22 4D 18` | lz4 frame |
| `50 41 52 31` (`PAR1`) | Parquet |

Three objects per container are sampled — first, middle, last of the §3 listing — via
ranged GETs, so only a few KiB are transferred per object.

In [ ]:
import string
import zlib

MAGICS = [
    (b"Obj\x01", "avro"),
    (b"\xd4\xc3\xb2\xa1", "pcap (usec LE)"),
    (b"\xa1\xb2\xc3\xd4", "pcap (usec BE)"),
    (b"\x4d\x3c\xb2\xa1", "pcap (nsec LE)"),
    (b"\xa1\xb2\x3c\x4d", "pcap (nsec BE)"),
    (b"\x0a\x0d\x0d\x0a", "pcapng"),
    (b"\x1f\x8b", "gzip"),
    (b"\x28\xb5\x2f\xfd", "zstd"),
    (b"\x04\x22\x4d\x18", "lz4"),
    (b"PAR1", "parquet"),
]

PRINTABLE = set(string.printable.encode())


def sniff_bytes(buf):
    for magic, name in MAGICS:
        if buf.startswith(magic):
            return name
    if buf and all(b in PRINTABLE for b in buf[:256]):
        return "text"
    return "unknown"


def head_bytes(bucket, key, n=4096):
    resp = s3.get_object(Bucket=bucket, Key=key, Range=f"bytes=0-{n - 1}")
    return resp["Body"].read()


def gunzip_head(buf, n=512):
    """Decompress as much of a (possibly truncated) gzip head as possible."""
    try:
        return zlib.decompressobj(wbits=31).decompress(buf, n)
    except zlib.error:
        return b""


def sniff_object(bucket, key):
    head = head_bytes(bucket, key)
    fmt = sniff_bytes(head)
    if fmt == "gzip":
        fmt = f"gzip -> {sniff_bytes(gunzip_head(head))}"
    if fmt == "unknown":
        fmt = f"unknown (first bytes: {head[:24].hex(' ')})"
    return fmt


def sample_keys(objs, n):
    keys = [o["Key"] for o in objs]
    if len(keys) <= n:
        return keys
    step = (len(keys) - 1) / (n - 1)
    return [keys[round(i * step)] for i in range(n)]


sniffed = {}
for bucket, objs in listings.items():
    print(f"=== {bucket} ===")
    results = []
    for key in sample_keys(objs, N_SAMPLE_OBJECTS):
        fmt = sniff_object(bucket, key)
        size = next(o["Size"] for o in objs if o["Key"] == key)
        results.append((key, size, fmt))
        print(f"  {fmt:<28} {size / 2**20:9.2f} MiB  {key}")
    sniffed[bucket] = results
    print()

## 5 · Confirm the FlowTuple format

For the first Avro object found in each container, stream the object just far enough
to read the **embedded writer schema** and the first few records (fastavro reads the
boto3 body lazily, so only the leading block is downloaded). The verdict checks that
the schema contains the core FlowTuple fields the darknet module relies on:

`time · src_ip · packet_cnt · prefix2asn · netacq_country`

(The real FlowTuple v4 schema has more fields than these — the full field list is
printed so we see exactly what the greynet records carry.)

Text objects (the greynet address list, most likely) get their first lines shown;
raw pcap objects are reported as such — a pcap is *not* FlowTuple, and finding one
here would mean the flow aggregation step still lives with us.

In [ ]:
import gzip
import ipaddress

import fastavro
import pandas as pd

# The fields the darknet module (nids-ucsdnt-expanse) relies on in FlowTuple v4 records.
FLOWTUPLE_CORE_FIELDS = {"time", "src_ip", "packet_cnt", "prefix2asn", "netacq_country"}


def inspect_avro(bucket, key, n_records, gzipped=False):
    """Stream an Avro object: (schema name, field names, first records)."""
    body = s3.get_object(Bucket=bucket, Key=key)["Body"]
    stream = gzip.GzipFile(fileobj=body) if gzipped else body
    try:
        reader = fastavro.reader(stream)
        schema = reader.writer_schema
        fields = [f["name"] for f in schema.get("fields", [])]
        records = []
        for rec in reader:
            records.append(rec)
            if len(records) >= n_records:
                break
    finally:
        body.close()
    return schema.get("name", "?"), fields, records


def records_frame(records):
    df = pd.DataFrame(records)
    if "src_ip" in df and pd.api.types.is_integer_dtype(df["src_ip"]):
        df["src_ip"] = df["src_ip"].map(lambda v: str(ipaddress.ip_address(int(v) & 0xFFFFFFFF)))
    if "time" in df:
        df.insert(0, "time_utc", pd.to_datetime(df["time"], unit="s", utc=True, errors="coerce"))
    return df


def show_text_head(bucket, key, n_lines=15):
    head = head_bytes(bucket, key, 8192)
    if head[:2] == b"\x1f\x8b":
        head = gunzip_head(head, 8192)
    lines = head.decode("utf-8", errors="replace").splitlines()
    print("\n".join(f"    {line}" for line in lines[:n_lines]))
    if len(lines) > n_lines:
        print(f"    ... ({len(lines)} lines in the first 8 KiB alone)")


flowtuple_confirmed = {}
record_frames = {}

for bucket, results in sniffed.items():
    print(f"=== {bucket} ===")
    flowtuple_confirmed[bucket] = False
    avro_done = text_done = False
    for key, size, fmt in results:
        if "avro" in fmt and not avro_done:
            avro_done = True
            name, fields, records = inspect_avro(
                bucket, key, N_AVRO_RECORDS, gzipped=fmt.startswith("gzip")
            )
            missing = FLOWTUPLE_CORE_FIELDS - set(fields)
            ok = not missing
            flowtuple_confirmed[bucket] |= ok
            print(f"  {key}")
            print(f"    schema : {name}  ({len(fields)} fields)")
            print(f"    fields : {', '.join(fields)}")
            verdict = "YES" if ok else f"NO — missing {sorted(missing)}"
            print(f"    FlowTuple core fields present: {verdict}")
            record_frames[bucket] = records_frame(records)
            display(record_frames[bucket])
        elif fmt.endswith("text") and not text_done:
            text_done = True
            print(f"  {key} — text; first lines:")
            show_text_head(bucket, key)
        elif "pcap" in fmt:
            print(f"  {key} — raw packet capture ({fmt}), not FlowTuple.")
    print()

## 6 · Verdict

In [ ]:
rows = []
for bucket in CONTAINERS:
    fmts = sorted({fmt for _, _, fmt in sniffed.get(bucket, [])})
    rows.append(
        {
            "container": bucket,
            "objects_listed": len(listings.get(bucket, [])),
            "formats_seen": ", ".join(fmts) or "-",
            "flowtuple_confirmed": flowtuple_confirmed.get(bucket, False),
        }
    )
verdict = pd.DataFrame(rows).set_index("container")
display(verdict)

## Next steps

- If a container's Avro schema checked out, the darknet module's Spark pattern carries
  over directly: `spark-avro` + `hadoop-aws`, `s3a://` paths against the same endpoint
  with `path.style.access=true`, then per-day filter/aggregate to Parquet.
- If `telescope-sdscnt-pcap-live` turned out to be raw pcap, flow aggregation happens
  on our side first — the `nids-telescope-traffic` module's dpkt streaming reader is
  the starting point.
- `sdscgreynetlist` (the greynet address list) is what lets us normalize greynet
  traffic by the number of monitored addresses when comparing against the darknet.

If §3 failed outright: wrong endpoint → set `ENDPOINT_OVERRIDE` in §1; access denied →
check the variable names §2 printed against the env file.